# PhoBERT NER — Colab cho team (mỗi người tự train)

`Runtime → Change runtime type → T4 GPU` trước khi chạy. Chạy cell theo thứ tự từ trên xuống.

In [ ]:
#@title 1. Clone repo + cd vào project
!git clone https://github.com/letuanlocc/training_model.git /content/training_model
%cd /content/training_model/phobert_ner
!ls

In [ ]:
#@title 2. Tạo nhánh riêng của bạn
# Đổi "ten/data" thành tên bạn, ví dụ "an/data"
!git checkout -b ten/data

In [ ]:
#@title 3. Cài môi trường (Java + Python packages + VnCoreNLP)
!apt-get -qq update && apt-get -qq install -y openjdk-17-jre-headless > /dev/null

!grep -v '^#' requirements_working.txt | grep -v -E '^(numpy|pandas|matplotlib|scikit-learn)' | sed '/^$/d' > /tmp/req_colab.txt
!pip -q install -r /tmp/req_colab.txt

import os
VN = os.path.abspath('vncorenlp'); B = 'https://raw.githubusercontent.com/vncorenlp/VnCoreNLP/master'
os.makedirs(f'{VN}/models/wordsegmenter', exist_ok=True)
for f in ['VnCoreNLP-1.2.jar', 'VnCoreNLP-1.1.1.jar', 'models/wordsegmenter/vi-vocab', 'models/wordsegmenter/wordsegmenter.rdr']:
    !wget -q -nc -O {VN}/{f} {B}/{f}
print('Xong.')

## 4. Tự tạo file dữ liệu của bạn

Mở panel **Files** bên trái Colab → `training_model/phobert_ner/data_build/` → chuột phải → **New file** →
đặt tên `handwritten_<ten>.txt` → viết câu, mỗi dòng 1 câu, đánh dấu thực thể bằng `[text|TAG]`
(TAG là `SKILL`/`TIME`/`LOC`). Xem ví dụ có sẵn trong `handwritten.txt` hoặc đọc `CONTRIBUTING.md`.

Lưu file xong mới chạy tiếp bước dưới.

## 5. Sinh dữ liệu + Train

Chọn 1 kiến trúc (softmax nhanh nhất, crf/lstm_crf thường chính xác hơn). ~15-30 phút trên GPU T4.

In [ ]:
ARCH = "softmax"  #@param ["softmax", "crf", "lstm_crf"]

!python data_build/build_data.py
!python data_build/validate_data.py
!bash train_{ARCH}.sh 2>&1 | grep -v 'it/s]'

## 6. Lưu model lên Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Đổi "ten" thành tên bạn
!mkdir -p "/content/drive/MyDrive/phobert_ner_outputs"
!cp -r outputs/{ARCH} "/content/drive/MyDrive/phobert_ner_outputs/{ARCH}_ten"

## 7. Push dữ liệu lên git (lên nhánh của bạn)

Chỉ push file `handwritten_<ten>.txt` — KHÔNG push `data_dir/`, `outputs/` (đã có trong `.gitignore`).

In [ ]:
# Đổi tên/email + tên file cho đúng
!git config user.name "Tên thật của bạn"
!git config user.email "email_github_cua_ban@example.com"
!git add data_build/handwritten_ten.txt
!git commit -m "Thêm câu dữ liệu"
!git push -u origin HEAD

`git push` hỏi tài khoản/mật khẩu → dùng username GitHub + **Personal Access Token** (không dùng mật khẩu thường),
và bạn phải là collaborator của repo. Sau khi push, tạo Pull Request từ nhánh của bạn vào `main` trên GitHub.

## 8. (Tuỳ chọn) Thử model với câu của bạn

In [ ]:
CAU_CAN_THU = "tôi giỏi tiếng anh, rảnh vào tối và ở Hải Châu"  #@param {type:"string"}

import subprocess
cmd = f'printf "{CAU_CAN_THU}\\n" | python VPhoBertTagger/main.py predict --model_path outputs/{ARCH}/best_model.pt'
out = subprocess.run(cmd, shell=True, capture_output=True, text=True)
for line in out.stdout.splitlines():
    if 'EOFError' not in line:
        print(line)

---
### Lỗi hay gặp
- `ModuleNotFoundError` bất kỳ module nào / `vncorenlp jar not found` / Colab tự ngắt kết nối giữa buổi
  → chạy lại cell 1, 2, 3 (runtime bị reset, mất hết file + package đã cài).
- `git push` báo lỗi quyền / authentication failed
  → bạn chưa được add làm collaborator trên GitHub repo, hoặc dùng mật khẩu thường thay vì Personal Access Token.